# 8 · Scenario sweeps + visualizations — distributed SeQUeNCe BB84

Run the `qne-sequence` emulator across a **matrix of scenarios**, capture results to
JSON, and render figures. Two parts:

- **A · Local sweep** (no testbed) — fidelity / distance / efficiency / throughput
  sweeps over loopback (TCP descriptor transport). Fast, reproducible, slice-free —
  the validation curves (QBER vs F, Shor-Preskill cutoff, key-yield vs distance).
- **B · On-testbed sweep** (FABRIC raw/P4, optional) — re-run a distance sweep with real
  `0x7101` photons through the P4 switch, plus a **classical-network-effects** sweep
  (latency/jitter/loss on TCP only) — the QFabric research lever, over the SeQUeNCe stack.

Results → `qne-sequence/results/*.json`; figures → `qne-sequence/results/figures/`.

## A · Local scenario sweep (no testbed)

Runs anywhere the repo `.venv` (with `sequence==1.0.0` + numpy + matplotlib) is the
kernel — e.g. your dev box, or the SPHERE/FABRIC XDC if that env is installed.

In [ ]:
import sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
QNE_SEQ = PROJECT_DIR / 'qne-sequence'
for p in (str(PROJECT_DIR), str(PROJECT_DIR / 'scripts'), str(QNE_SEQ)):
    if p not in sys.path:
        sys.path.insert(0, p)

import sweep, plots   # qne-sequence/sweep.py, qne-sequence/plots.py

QUICK = False    # True = small/fast smoke matrix; False = full matrix (~1 min)

In [ ]:
REPS = 5   # repetitions per point -> mean +/- std (error bars)
rows = sweep.run_sweep(sweep.default_matrix(quick=QUICK), reps=REPS)
out = QNE_SEQ / 'results' / 'sequence_scenarios.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(rows, indent=2))
print(f"\n{len(rows)} points x{REPS} reps, {sum(1 for r in rows if r.get('key'))} formed a key -> {out}")

In [ ]:
# render every figure inline (auto-covers all sweep axes) + save PNGs
from IPython.display import display
for name, fn in plots._FIGS.items():
    print(f"--- {name} ---")
    display(fn(rows))

saved = plots.save_all(out, QNE_SEQ / 'results' / 'figures')
print('\nsaved figures:')
for p in saved:
    print('  ', p)

**Reading the figures** (each point is mean ± std over `REPS` runs)
- *QBER vs fidelity* — sampled QBER tracks the analytical $(1-F)/2$; all points well under the 11% threshold.
- *Secure fraction vs QBER* — points fall on the Shor-Preskill $1-2H(Q)$ curve, hitting 0 at ~11%.
- *Key yield vs distance / attenuation* — sifted/secure bits fall as fiber loss $1-10^{-\alpha L/10}$ rises.
- *Key yield vs efficiency* — detector efficiency scales the sifted count.
- *QBER vs dark-count rate* — flat until the rate is high enough to add false detections (the dark-count floor).
- *QBER vs sample fraction* — mean ~flat; the **error bar shrinks** as more bits are disclosed (estimator variance ↓).
- *Key formation vs target length* — flat secure-bit yield until the target exceeds the sifted budget (dotted = failed).
- *Throughput* — `bulk` amortizes per-photon framing; `per_event` keeps per-photon fidelity at a lower ceiling.

**Coverage / limits:** the sweep varies every *behavioral* knob one axis at a time
(fidelity, distance, attenuation, efficiency, dark-count rate, sample fraction, key
length, pulse count, photon mode). Not swept here: `quantum_transport=raw` (real P4
photons — §B below), `time_scale`/`channel_delay` (affect only wall-clock pacing), and
`key_num` (>1 key per request is **not supported** by `DistributedBB84`).

## B · On-testbed sweep (FABRIC) — optional

**Prereqs:** run **notebook 1** (slice + data-plane IPs; BMv2 if using the switch) and
**notebook 7** once (`setup_sequence_runtime` built `.venv-qne` on both nodes). This
re-runs the distance sweep on real nodes, then sweeps classical-channel conditions.

Pick the channel mode with `TRANSPORT`/`LOSS` (cell below): `raw`+`switch` uses the P4
switch (default); `raw`+`model`, `tcp`, or `loss='none'` run **switch-free**. The
`configure_switch` step is skipped automatically when no switch is needed.

In [ ]:
SLICE_NAME    = 'qfabric-bb84-2'
TRANSPORT     = 'raw'    # 'raw' (0x7101 frames) | 'tcp' (descriptors over TCP, no switch)
LOSS          = 'switch' # 'switch' (BMv2 P4) | 'model' (software, no switch) | 'none' (lossless) | 'auto'
ATTEN         = 0.2      # dB/km
FIDELITY      = 0.98
EFFICIENCY    = 0.8
DARK          = 10.0
KEY_LENGTH    = 256
SAMPLE_FRAC   = 0.2
FAB_PULSES    = 20000
DRAIN_MS      = 500
FAB_DISTANCES = [1, 10, 25, 50]          # km (sets the loss per point)
NETEM_DISTANCE = 10                       # km, fixed scenario for the network-effects sweep

# the P4 switch is only needed for raw + switch/auto; tcp / model / none run switch-free
USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))

import deploy_fabric as df
fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()
print(f"transport={TRANSPORT} loss={LOSS} -> {'using P4 switch' if USE_SWITCH else 'switch-free'}")

In [ ]:
# B.1 — distance sweep (real 0x7101 photons; loss via P4 switch or software per LOSS)
fab_rows = []
for d in FAB_DISTANCES:
    if USE_SWITCH:
        df.configure_switch(slice_obj, int(df.loss_probability(d, ATTEN) * (2**32)))
    a, b = df.run_sequence_bb84(
        slice_obj, transport=TRANSPORT, loss=LOSS,
        num_pulses=FAB_PULSES, key_length=KEY_LENGTH,
        fidelity=FIDELITY, efficiency=EFFICIENCY, dark_count_rate=DARK,
        distance_km=d, attenuation=ATTEN, sample_fraction=SAMPLE_FRAC,
        photon_mode='bulk', photon_drain_ms=DRAIN_MS)
    if a:
        a['sweep'], a['x'], a['key'] = 'distance', d, (a.get('key') is not None)
        fab_rows.append(a)

fab_out = QNE_SEQ / 'results' / 'sequence_scenarios_fabric.json'
fab_out.write_text(json.dumps(fab_rows, indent=2))
print(f"saved {len(fab_rows)} FABRIC distance points -> {fab_out}")
display(plots.fig_distance(fab_rows))

In [ ]:
# B.2 — classical-network-effects sweep (impairs TCP:5100 only; photon plane untouched)
CONDITIONS = [
    {'name': 'baseline'},
    {'name': 'latency_50ms',  'delay_ms': 50},
    {'name': 'latency_150ms', 'delay_ms': 150},
    {'name': 'jitter_50pm20', 'delay_ms': 50, 'jitter_ms': 20},
    {'name': 'loss_1pct',     'loss_pct': 1.0},
]

if USE_SWITCH:
    df.configure_switch(slice_obj, int(df.loss_probability(NETEM_DISTANCE, ATTEN) * (2**32)))
net_rows = []
try:
    for cond in CONDITIONS:
        df.clear_classical_netem(slice_obj)
        kw = {k: v for k, v in cond.items() if k != 'name'}
        if kw:
            df.apply_classical_netem(slice_obj, **kw)
        a, b = df.run_sequence_bb84(
            slice_obj, transport=TRANSPORT, loss=LOSS,
            num_pulses=FAB_PULSES, key_length=KEY_LENGTH,
            fidelity=FIDELITY, efficiency=EFFICIENCY, dark_count_rate=DARK,
            distance_km=NETEM_DISTANCE, attenuation=ATTEN,
            sample_fraction=SAMPLE_FRAC, photon_mode='bulk', photon_drain_ms=DRAIN_MS)
        row = {'condition': cond['name'], **kw}
        if a:
            el = a.get('elapsed_s') or 0.0
            fk = a.get('final_key_bits') or 0
            row.update({'qber': a.get('qber'), 'sifted_bits': a.get('sifted_bits'),
                        'final_key_bits': fk, 'elapsed_s': el,
                        'key_bits_per_s': (fk / el) if el > 0 else 0.0,
                        'key': a.get('key') is not None})
        net_rows.append(row)
finally:
    df.clear_classical_netem(slice_obj)

net_out = QNE_SEQ / 'results' / 'sequence_network_effects.json'
net_out.write_text(json.dumps(net_rows, indent=2))
print(f"saved {len(net_rows)} conditions -> {net_out}")
display(plots.fig_network_effects(net_rows))

**Interpreting B** — QBER should stay ~flat across classical conditions (TCP is reliable,
so sifting integrity is preserved), while **time-to-key** grows and **key-bits/s** drops
with latency/jitter/loss. That gap — invisible to ideal-channel simulators — is the
QFabric contribution, now measured over the full SeQUeNCe BB84 stack.

Cleanup: `df.cleanup(fablib, SLICE_NAME)` deletes the slice.